In [3]:
from tools.files import edit_file, list_files, read_file


TOOL_REGISTRY = {
    "read_file": read_file,
    "list_files": list_files,
    "edit_file": edit_file,
}

In [4]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": (
                "Reads the full contents of a text file. "
                "Use this tool before editing a file or when you need to inspect "
                "the implementation of a specific module. "
                "The path must point to an existing text file."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": (
                            "Path to the file that should be read. "
                            "Prefer a path relative to the project root, "
                            "for example: 'src/auth.py'."
                        ),
                    }
                },
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": (
                "Lists files and directories recursively starting from the specified "
                "directory. Use this tool to inspect the project structure and find "
                "files that may be relevant to the user's task. "
                "Ignored directories are excluded automatically."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "root": {
                        "type": "string",
                        "description": (
                            "Directory from which the recursive listing should start. "
                            "Use '.' to inspect the project root."
                        ),
                    },
                    "max_depth": {
                        "type": "integer",
                        "description": (
                            "Maximum recursion depth. Use a small value such as 2 or 3 "
                            "for an initial project overview. Increase it only when "
                            "deeper inspection is necessary."
                        ),
                        "minimum": 0,
                        "maximum": 10,
                        "default": 3,
                    },
                },
                "required": ["root"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "edit_file",
            "description": (
                "Replaces the entire contents of a text file with new content. "
                "Always read the file with read_file before using this tool. "
                "The content argument must contain the complete final contents of "
                "the file, not only the changed fragment. "
                "Use this tool only when an actual file modification is required."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": (
                            "Path to the file that should be overwritten. "
                            "Prefer a path relative to the project root, "
                            "for example: 'src/auth.py'."
                        ),
                    },
                    "content": {
                        "type": "string",
                        "description": (
                            "Complete new contents of the file. "
                            "This value fully replaces the existing file contents, "
                            "so all unchanged code must also be included."
                        ),
                    },
                },
                "required": ["path", "content"],
                "additionalProperties": False,
            },
        },
    },
]


SYSTEM_PROMPT = """
You are a coding agent working with a software project.

Use the provided tools to inspect and modify project files.

Rules:
1. Inspect the project structure before assuming where relevant code is located.
2. Never invent the contents of a file that you have not read.
3. Always use read_file before editing an existing file.
4. The edit_file tool replaces the entire file, so its content argument must include
   the complete final file contents.
5. Make only changes required for the user's task.
6. Do not claim that a file was modified until edit_file reports success.
7. If a tool returns an error, analyze the error and choose a safe next action.
8. When the task is complete, respond normally without calling another tool.
"""


In [45]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY=os.getenv('API_KEY')
BASE_URL=os.getenv('BASE_URL')

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
)

messages = [
    {'role':'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': 'Write a /health endpoint in dir working_directory'}
]


resp = client.chat.completions.create(
    model="deepseek/deepseek-v4-flash",        # или anthropic/claude-3.5-sonnet
    messages=messages,
    tools=TOOLS
)

print(resp)

ChatCompletion(id='gen-1784667400-nopg4RkP3cvjD0ggTt2y', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_74ca14f9621e40aa8bd67d23', function=Function(arguments='{"root": ".", "max_depth": 3}', name='list_files'), type='function', index=0)], reasoning='Let me first inspect the project structure to understand what kind of project this is and where to add the health endpoint.', reasoning_details=[{'type': 'reasoning.text', 'text': 'Let me first inspect the project structure to understand what kind of project this is and where to add the health endpoint.', 'format': 'unknown', 'index': 0}]), native_finish_reason='tool_calls')], created=1784667400, model='deepseek/deepseek-v4-flash', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsag

In [46]:
# получение первого ответа от модели. количество choices зависит от аргумента "n" в client.chat.completions.create
resp.choices[0]

Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_74ca14f9621e40aa8bd67d23', function=Function(arguments='{"root": ".", "max_depth": 3}', name='list_files'), type='function', index=0)], reasoning='Let me first inspect the project structure to understand what kind of project this is and where to add the health endpoint.', reasoning_details=[{'type': 'reasoning.text', 'text': 'Let me first inspect the project structure to understand what kind of project this is and where to add the health endpoint.', 'format': 'unknown', 'index': 0}]), native_finish_reason='tool_calls')

In [47]:
# получение причины остановки модели
resp.choices[0].finish_reason

'tool_calls'

In [48]:
# вывод информации о сообщении модели
resp.choices[0].message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_74ca14f9621e40aa8bd67d23', function=Function(arguments='{"root": ".", "max_depth": 3}', name='list_files'), type='function', index=0)], reasoning='Let me first inspect the project structure to understand what kind of project this is and where to add the health endpoint.', reasoning_details=[{'type': 'reasoning.text', 'text': 'Let me first inspect the project structure to understand what kind of project this is and where to add the health endpoint.', 'format': 'unknown', 'index': 0}])

In [49]:
# вывод текста модели
resp.choices[0].message.content

In [50]:
# reasoning
resp.choices[0].message.reasoning

'Let me first inspect the project structure to understand what kind of project this is and where to add the health endpoint.'

In [51]:
import json

In [52]:
json.loads('{"max_depth": 4, "root": "."}')

{'max_depth': 4, 'root': '.'}

In [53]:
tools = []

if resp.choices[0].finish_reason == 'tool_calls':
    for toolcall in resp.choices[0].message.tool_calls:
        print(toolcall.id)
        print(toolcall.function)

        json_data = json.loads(toolcall.function.arguments)
        function = TOOL_REGISTRY.get(toolcall.function.name, False)

        if function:
            print('\nFUNCTION OUTPUT:')
            print(function(**json_data))

            messages.append({
                'role': 'tool',
                'tool_call_id': toolcall.id,
                'content': str(function(**json_data))
            })
            
        print('-'*100)

        

call_74ca14f9621e40aa8bd67d23
Function(arguments='{"root": ".", "max_depth": 3}', name='list_files')

FUNCTION OUTPUT:
[{'path': 'tools', 'type': 'directory'}, {'path': 'tools/files.py', 'type': 'file'}, {'path': 'tools/__init__.py', 'type': 'file'}, {'path': 'tools/schemas.py', 'type': 'file'}, {'path': '.DS_Store', 'type': 'file'}, {'path': 'config.py', 'type': 'file'}, {'path': 'handle_testing.ipynb', 'type': 'file'}, {'path': 'uv.lock', 'type': 'file'}, {'path': 'working_directory', 'type': 'directory'}, {'path': 'working_directory/main.py', 'type': 'file'}, {'path': 'tools.py', 'type': 'file'}, {'path': 'pyproject.toml', 'type': 'file'}, {'path': 'agent', 'type': 'directory'}, {'path': 'agent/llm.py', 'type': 'file'}, {'path': 'agent/loop.py', 'type': 'file'}, {'path': 'README.md', 'type': 'file'}, {'path': '.gitignore', 'type': 'file'}, {'path': '.env', 'type': 'file'}, {'path': '.python-version', 'type': 'file'}, {'path': 'tests.py', 'type': 'file'}, {'path': 'main.py', 'type': 

In [54]:
messages

[{'role': 'system',
  'content': "\nYou are a coding agent working with a software project.\n\nUse the provided tools to inspect and modify project files.\n\nRules:\n1. Inspect the project structure before assuming where relevant code is located.\n2. Never invent the contents of a file that you have not read.\n3. Always use read_file before editing an existing file.\n4. The edit_file tool replaces the entire file, so its content argument must include\n   the complete final file contents.\n5. Make only changes required for the user's task.\n6. Do not claim that a file was modified until edit_file reports success.\n7. If a tool returns an error, analyze the error and choose a safe next action.\n8. When the task is complete, respond normally without calling another tool.\n"},
 {'role': 'user',
  'content': 'Write a /health endpoint in dir working_directory'},
 {'role': 'tool',
  'tool_call_id': 'call_74ca14f9621e40aa8bd67d23',
  'content': "[{'path': 'tools', 'type': 'directory'}, {'path'

In [55]:
resp = client.chat.completions.create(
    model="deepseek/deepseek-v4-flash",        # или anthropic/claude-3.5-sonnet
    messages=messages,
    tools=TOOLS
)

print(resp)

ChatCompletion(id='gen-1784667440-2mPrHtWeH8W1t6SteJyO', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_gzyion394li6sa6mo4u9u9pf', function=Function(arguments='{"path": "working_directory/main.py"}', name='read_file'), type='function', index=0)], reasoning="Let me look at the working_directory to understand what's already there.", reasoning_details=[{'type': 'reasoning.text', 'text': "Let me look at the working_directory to understand what's already there.", 'format': 'unknown', 'index': 0}]), native_finish_reason='tool_calls')], created=1784667440, model='deepseek/deepseek-v4-flash', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=61, prompt_tokens=1175, total_tokens=1236, completion_tokens_details=Comple

In [56]:
resp.choices[0].message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_gzyion394li6sa6mo4u9u9pf', function=Function(arguments='{"path": "working_directory/main.py"}', name='read_file'), type='function', index=0)], reasoning="Let me look at the working_directory to understand what's already there.", reasoning_details=[{'type': 'reasoning.text', 'text': "Let me look at the working_directory to understand what's already there.", 'format': 'unknown', 'index': 0}])

In [2]:
import json
json.loads('{"a": 2')

JSONDecodeError: Expecting ',' delimiter: line 1 column 8 (char 7)

In [ ]:
from json import JSONDecodeError